# 📊 ANÁLISE ESTATÍSTICA DESCRITIVA - PYTHON
## Impacto da Presença Digital em Hotéis Brasileiros

---

**Autor:** Matheus Folle  
**Dataset:** 10.316 hotéis brasileiros  
**Objetivo:** Comparar performance de hotéis COM vs SEM website  

---

### Estrutura do Notebook:
1. Setup e Importações
2. Carregamento e Preparação dos Dados
3. Visualizações Python (7 gráficos)
4. Análise de Correlação
5. Resumo Final

**IMPORTANTE:** Execute as células NA ORDEM (não use "Run All" de uma vez)

## 1️⃣ SETUP E CONFIGURAÇÕES

In [ ]:
# 1.1 Instalar dependências
!pip install pandas numpy matplotlib seaborn scipy openpyxl -q
print("✅ Bibliotecas instaladas!")

In [ ]:
# 1.2 Importar bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

# Configurações de visualização
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

# Paletas de cores (azul escuro + verde escuro)
COLOR_COM = '#2C5F7C'      # Azul escuro (COM website)
COLOR_SEM = '#4A7C59'      # Verde escuro (SEM website)
PALETTE_BINARY = [COLOR_SEM, COLOR_COM]  # Para has_website binário
PALETTE_CREST = "crest"    # Degradê azul-verde

print("✅ Bibliotecas importadas e configurações aplicadas!")

## 2️⃣ CARREGAMENTO E PREPARAÇÃO DOS DADOS

In [ ]:
# 2.1 Carregar dataset
# ATENÇÃO: Faça upload do arquivo 'data-hoteis-atualizado.xlsx' no Colab antes de executar

file_path = 'data-hoteis-atualizado.xlsx'

try:
    df = pd.read_excel(file_path)
    print(f"✅ Arquivo '{file_path}' carregado com sucesso!")
    print(f"📊 Total de registros: {len(df):,}")
    print(f"📋 Colunas: {df.columns.tolist()}")
except FileNotFoundError:
    print(f"❌ ERRO: Arquivo '{file_path}' não encontrado.")
    print("💡 Faça upload do arquivo para o Colab.")
except Exception as e:
    print(f"❌ Erro: {e}")

In [ ]:
# 2.2 Inspeção inicial
print("\n--- 📋 PRIMEIRAS 5 LINHAS ---")
display(df.head())

print("\n--- 📊 INFORMAÇÕES DO DATASET ---")
df.info()

print("\n--- 🔢 VALORES ÚNICOS DE has_website ---")
print(df['has_website'].value_counts())

In [ ]:
# 2.3 CORREÇÃO CRÍTICA: Converter has_website para binário numérico
# Se a coluna contém 'SEM'/'COM' (texto), converter para 0/1

if df['has_website'].dtype == 'object':  # Se é texto
    print("⚠️ Detectado has_website como TEXTO. Convertendo para numérico...")
    # Mapear 'SEM' → 0, 'COM' → 1
    df['has_website'] = df['has_website'].map({'SEM': 0, 'COM': 1})
    print("✅ Conversão concluída!")
    print(df['has_website'].value_counts())
else:
    print("✅ has_website já é numérico.")

# Verificar valores ausentes
print(f"\n🔍 Valores nulos: {df['has_website'].isnull().sum()}")

In [ ]:
# 2.4 Estatísticas gerais
print("\n--- 📉 ESTATÍSTICAS GERAIS ---")
display(df[['totalScore', 'reviewsCount']].describe())

# Identificar hotéis sem avaliação (outliers)
hoteis_sem_avaliacao = (df['totalScore'] <= 0.2).sum()
print(f"\n⚠️ Hotéis com totalScore ≤ 0.2 (sem avaliação): {hoteis_sem_avaliacao:,}")
print(f"   Percentual: {(hoteis_sem_avaliacao / len(df) * 100):.2f}%")

## 3️⃣ VISUALIZAÇÕES PYTHON (7 GRÁFICOS)

**Nota:** Estatísticas descritivas (média, mediana, etc.) já calculadas no Excel. Gráficos complementam a análise visual.

### 📊 GRÁFICO 1: Barras - Média de Reviews

In [ ]:
# Gráfico 1: Comparação da média de reviewsCount
plt.figure(figsize=(10, 7))
sns.barplot(x='has_website', y='reviewsCount', data=df,
            palette=PALETTE_BINARY, estimator=np.mean, ci=None)

plt.title('Gráfico 1 (Python): Média de reviewsCount por Presença Digital', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Tem Website?', fontsize=13)
plt.ylabel('Média de Número de Reviews', fontsize=13)
plt.xticks([0, 1], ['SEM Website', 'COM Website'])

# Valores nas barras
media_sem = df[df['has_website'] == 0]['reviewsCount'].mean()
media_com = df[df['has_website'] == 1]['reviewsCount'].mean()
plt.text(0, media_sem + 15, f'{media_sem:.1f}', ha='center', fontsize=12, fontweight='bold')
plt.text(1, media_com + 15, f'{media_com:.1f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('grafico_1_barras_reviews.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"📈 Diferença: +{((media_com / media_sem - 1) * 100):.1f}% mais reviews (COM vs SEM)")

### 📊 GRÁFICO 2: Dispersão por Grupo

In [ ]:
# Gráfico 2: Scatter plot
# Filtro: Excluir hotéis sem avaliação
df_plot = df[(df['totalScore'] > 0.2) & (df['reviewsCount'] > 0)]

plt.figure(figsize=(14, 9))
sns.scatterplot(x='reviewsCount', y='totalScore', hue='has_website', 
                data=df_plot, palette=PALETTE_BINARY, alpha=0.6, s=40)

plt.xscale('log')
plt.title('Gráfico 2 (Python): Dispersão de reviewsCount vs totalScore por Grupo', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Número de Reviews (escala logarítmica)', fontsize=13)
plt.ylabel('Nota Total (totalScore)', fontsize=13)
plt.legend(title='Presença Digital', labels=['SEM Website', 'COM Website'], fontsize=11)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('grafico_2_dispersao_grupo.png', dpi=300, bbox_inches='tight')
plt.show()

### 📊 GRÁFICO 3: Violino - Distribuição de Score

In [ ]:
# Gráfico 3: Violin Plot
plt.figure(figsize=(12, 8))
sns.violinplot(x='has_website', y='totalScore', 
               data=df[df['totalScore'] > 0.2],
               palette=PALETTE_BINARY, inner='quartile')

plt.title('Gráfico 3 (Python): Distribuição de totalScore (COM vs SEM)', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Presença Digital', fontsize=13)
plt.ylabel('Nota Total (Score)', fontsize=13)
plt.xticks([0, 1], ['SEM Website', 'COM Website'])

plt.tight_layout()
plt.savefig('grafico_3_violino_score.png', dpi=300, bbox_inches='tight')
plt.show()

### 📊 GRÁFICO 4: ECDF - Curva de Percentil

In [ ]:
# Gráfico 4: Empirical Cumulative Distribution Function
plt.figure(figsize=(14, 8))

# Zoom até 2000 reviews
df_zoom = df[df['reviewsCount'] <= 2000]

sns.ecdfplot(data=df_zoom, x='reviewsCount', hue='has_website', 
             palette=PALETTE_BINARY, linewidth=2.5)

plt.title('Gráfico 4 (Python): Curva de Percentil (ECDF) de reviewsCount', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Número de Reviews', fontsize=13)
plt.ylabel('Proporção Acumulada (Percentil)', fontsize=13)
plt.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.5)

# Marcar Q3 do grupo SEM
p75_sem = df[df['has_website'] == 0]['reviewsCount'].quantile(0.75)
plt.axvline(x=p75_sem, color='gray', linestyle='--', alpha=0.7)
plt.axhline(y=0.75, color='gray', linestyle='--', alpha=0.7)

plt.legend(title='Presença Digital', labels=['SEM Website', 'COM Website'])
plt.tight_layout()
plt.savefig('grafico_4_ecdf_reviews.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"📊 Q3 SEM Website: {p75_sem:.0f} reviews")

### 📊 GRÁFICO 5: JointPlot - Densidade Hexagonal

In [ ]:
# Gráfico 5: JointPlot
g = sns.jointplot(
    data=df_plot,
    x='reviewsCount',
    y='totalScore',
    kind='hex',
    cmap=PALETTE_CREST,
    gridsize=40,
    height=10
)

g.fig.suptitle('Gráfico 5 (Python): Relação de Densidade (reviewsCount x totalScore)', 
               y=1.02, fontsize=16, fontweight='bold')
g.set_axis_labels('Número de Reviews', 'Nota Total', fontsize=13)
g.ax_joint.set_xscale('log')

plt.savefig('grafico_5_jointplot_densidade.png', dpi=300, bbox_inches='tight')
plt.show()

### 📊 GRÁFICO 6: Histograma Comparativo

In [ ]:
# Gráfico 6: Histograma sobreposto COM vs SEM
plt.figure(figsize=(14, 8))

# Filtrar apenas hotéis avaliados
df_avaliados = df[df['totalScore'] > 0.2]

# Calcular médias ANTES de plotar
media_sem_score = df_avaliados[df_avaliados['has_website'] == 0]['totalScore'].mean()
media_com_score = df_avaliados[df_avaliados['has_website'] == 1]['totalScore'].mean()

# Histograma SEM
plt.hist(df_avaliados[df_avaliados['has_website'] == 0]['totalScore'], 
         bins=50, alpha=0.6, color=COLOR_SEM, label='SEM Website', edgecolor='black')

# Histograma COM
plt.hist(df_avaliados[df_avaliados['has_website'] == 1]['totalScore'], 
         bins=50, alpha=0.6, color=COLOR_COM, label='COM Website', edgecolor='black')

# Linhas verticais das médias
plt.axvline(media_sem_score, color=COLOR_SEM, linestyle='--', linewidth=2.5, 
            label=f'Média SEM: {media_sem_score:.2f}')
plt.axvline(media_com_score, color=COLOR_COM, linestyle='--', linewidth=2.5, 
            label=f'Média COM: {media_com_score:.2f}')

plt.title('Gráfico 6 (Python): Histograma Comparativo - Distribuição de totalScore', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Nota Total (totalScore)', fontsize=13)
plt.ylabel('Frequência (Número de Hotéis)', fontsize=13)
plt.grid(axis='y', alpha=0.3)
plt.legend(fontsize=11, loc='upper left')

plt.tight_layout()
plt.savefig('grafico_6_histograma_comparativo.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n📊 Médias de totalScore:")
print(f"   SEM Website: {media_sem_score:.3f}")
print(f"   COM Website: {media_com_score:.3f}")
print(f"   Diferença: +{((media_com_score - media_sem_score) / media_sem_score * 100):.2f}%")

### 📊 GRÁFICO 7: Heatmap de Correlação

In [ ]:
# Gráfico 7: Heatmap de correlação
plt.figure(figsize=(10, 8))

# Selecionar variáveis (garantir que has_website é numérico)
variaveis_analise = ['totalScore', 'reviewsCount', 'has_website']
df_corr = df[variaveis_analise].corr()

# Criar heatmap
sns.heatmap(df_corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            vmin=-1, vmax=1)

plt.title('Gráfico 7 (Python): Heatmap de Correlação (Pearson)', 
          fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('grafico_7_heatmap_correlacao.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n📊 Matriz de Correlação:")
print(df_corr)

## 4️⃣ ANÁLISE DE CORRELAÇÃO POR GRUPO

In [ ]:
# Correlação separada por grupo
df_sem = df[df['has_website'] == 0]
df_com = df[df['has_website'] == 1]

# Correlação SEM website
corr_sem, p_sem = pearsonr(df_sem['reviewsCount'], df_sem['totalScore'])

# Correlação COM website
corr_com, p_com = pearsonr(df_com['reviewsCount'], df_com['totalScore'])

print("\n📊 CORRELAÇÃO DE PEARSON (reviewsCount x totalScore):")
print(f"\n   SEM Website:")
print(f"      r = {corr_sem:.4f}")
print(f"      p-value = {p_sem:.6f} {'(significativo)' if p_sem < 0.05 else '(não significativo)'}")
print(f"\n   COM Website:")
print(f"      r = {corr_com:.4f}")
print(f"      p-value = {p_com:.6f} {'(significativo)' if p_com < 0.05 else '(não significativo)'}")
print(f"\n💡 Interpretação: Ambas as correlações são MUITO FRACAS (r < 0.2).")

## 5️⃣ RESUMO FINAL

In [ ]:
print("\n" + "="*60)
print("📋 RESUMO DA ANÁLISE ESTATÍSTICA DESCRITIVA")
print("="*60)

print(f"\n📊 DATASET:")
print(f"   Total: {len(df):,} hotéis")
print(f"   COM website: {(df['has_website'] == 1).sum():,} ({(df['has_website'] == 1).sum()/len(df)*100:.1f}%)")
print(f"   SEM website: {(df['has_website'] == 0).sum():,} ({(df['has_website'] == 0).sum()/len(df)*100:.1f}%)")

print(f"\n📈 PRINCIPAIS DESCOBERTAS:")
print(f"   1. Média de reviews: +{((media_com / media_sem - 1) * 100):.1f}% (COM vs SEM)")
print(f"   2. Correlação: r={corr_com:.4f} (COM) | r={corr_sem:.4f} (SEM) - MUITO FRACA")
print(f"   3. Diferença de score: +{((media_com_score - media_sem_score) / media_sem_score * 100):.2f}%")

print(f"\n✅ GRÁFICOS GERADOS (7):")
print(f"   • Gráfico 1: Barras (média reviews)")
print(f"   • Gráfico 2: Dispersão por grupo")
print(f"   • Gráfico 3: Violino (distribuição score)")
print(f"   • Gráfico 4: ECDF (percentil)")
print(f"   • Gráfico 5: JointPlot (densidade)")
print(f"   • Gráfico 6: Histograma comparativo")
print(f"   • Gráfico 7: Heatmap correlação")

print(f"\n🎯 CONCLUSÃO:")
print(f"   Hotéis COM website têm +{((media_com / media_sem - 1) * 100):.0f}% mais reviews e +{((media_com_score - media_sem_score) / media_sem_score * 100):.1f}% score.")
print(f"   Porém, correlação entre reviews e score é MUITO FRACA (r<0.2),")
print(f"   justificando Regressão Não-Linear na Entrega 2.")
print("\n" + "="*60)
print("✅ Análise concluída! Gráficos salvos em PNG (300 dpi).")
print("="*60)